###Sai Kadiravan S

# NLP Assignment 5: Token Classification — POS Tagging using DistilBERT

**Dataset:** Universal Dependencies (`commul/universal_dependencies`, `en_ewt`)  
**Model:** `distilbert-base-uncased`  
**Task:** POS Tagging using Token Classification

In [1]:
!pip install -q transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline
import evaluate


## Task 1 — Dataset Selection

We use **Universal Dependencies** for POS tagging because it is allowed in the assignment and avoids the old CoNLL dataset-script issue.

In [3]:
dataset = load_dataset('commul/universal_dependencies', 'en_ewt')
print(dataset)
print(dataset['train'][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

parquet/en_ewt/dev.parquet:   0%|          | 0.00/516k [00:00<?, ?B/s]

parquet/en_ewt/test.parquet:   0%|          | 0.00/523k [00:00<?, ?B/s]

parquet/en_ewt/train.parquet:   0%|          | 0.00/3.60M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/2001 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/12544 [00:00<?, ? examples/s]

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})
{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.', 'comments': ['newdoc id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000', '__SENT_ID__', 'newpar id = weblog-juancole.com_juancole_20

In [5]:
## Cell 4 — Inspect Dataset Features and POS Labels

print(dataset["train"].features)
print("\nSample row:\n", dataset["train"][0])

upos_feature = dataset["train"].features["upos"]
print("\nUPOS feature:", upos_feature)

{'sent_id': Value('string'), 'text': Value('string'), 'comments': List(Value('string')), 'tokens': List(Value('string')), 'lemmas': List(Value('string')), 'upos': List(ClassLabel(names=['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX'])), 'xpos': List(Value('string')), 'feats': List(Value('string')), 'head': List(Value('string')), 'deprel': List(Value('string')), 'deps': List(Value('string')), 'misc': List(Value('string')), 'mwt': List({'id': Value('string'), 'form': Value('string'), 'feats': Value('string'), 'misc': Value('string')}), 'empty_nodes': List({'id': Value('string'), 'form': Value('string'), 'lemma': Value('string'), 'upos': Value('string'), 'xpos': Value('string'), 'feats': Value('string'), 'head': Value('string'), 'deprel': Value('string'), 'deps': Value('string'), 'misc': Value('string')})}

Sample row:
 {'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 

In [6]:
## Cell 5 — Normalize POS Labels Safely

# Try to read label names directly from dataset metadata
upos_feature = dataset["train"].features["upos"]

if hasattr(upos_feature, "feature") and hasattr(upos_feature.feature, "names"):
    label_names = upos_feature.feature.names
elif hasattr(upos_feature, "names"):
    label_names = upos_feature.names
else:
    label_names = None

print("Label names from dataset metadata:", label_names)

def normalize_upos(example):
    tags = example["upos"]

    if len(tags) == 0:
        example["pos_tags"] = []
    elif isinstance(tags[0], str):
        example["pos_tags"] = tags
    elif label_names is not None:
        example["pos_tags"] = [label_names[t] for t in tags]
    else:
        raise ValueError("Could not infer UPOS label names from dataset metadata.")

    return example

dataset = dataset.map(normalize_upos)

print(dataset["train"][0]["tokens"])
print(dataset["train"][0]["pos_tags"])

Label names from dataset metadata: ['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


## Task 2 — Data Preprocessing

We tokenize with DistilBERT and align labels with tokens. Special tokens and non-first subword tokens get `-100`.

In [17]:
## Cell 6 — Create Label Mappings

all_pos_labels = sorted(
    {tag for split in dataset for ex in dataset[split]["pos_tags"] for tag in ex}
)

checkpoint = 'distilbert-base-uncased'

label2id = {label: i for i, label in enumerate(all_pos_labels)}
id2label = {i: label for label, i in label2id.items()}

print("POS Labels:", all_pos_labels)
print("Number of labels:", len(all_pos_labels))
print("label2id:", label2id)

POS Labels: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
Number of labels: 17
label2id: {'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}


In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)
    labels = []

    for i, word_labels in enumerate(examples['pos_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[word_labels[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print(tokenized_datasets)


Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'pos_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'pos_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'pos_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12544
    })
})


In [11]:
sample = tokenized_datasets['train'][0]
print('input_ids:', sample['input_ids'][:20])
print('attention_mask:', sample['attention_mask'][:20])
print('labels:', sample['labels'][:20])
print('tokens:', tokenizer.convert_ids_to_tokens(sample['input_ids'][:20]))


input_ids: [101, 2632, 1011, 23564, 2386, 1024, 2137, 2749, 2730, 21146, 28209, 14093, 2632, 1011, 2019, 2072, 1010, 1996, 14512, 2012]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
labels: [-100, 11, 12, 11, -100, 12, 0, 7, 15, 11, -100, 11, 11, 12, 11, -100, 12, 5, 7, 1]
tokens: ['[CLS]', 'al', '-', 'za', '##man', ':', 'american', 'forces', 'killed', 'sha', '##ikh', 'abdullah', 'al', '-', 'an', '##i', ',', 'the', 'preacher', 'at']


## Task 3 — Model Setup

In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    num_labels=len(all_pos_labels),
    id2label=id2label,
    label2id=label2id
)
print(model.config)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "ADJ",
    "1": "ADP",
    "2": "ADV",
    "3": "AUX",
    "4": "CCONJ",
    "5": "DET",
    "6": "INTJ",
    "7": "NOUN",
    "8": "NUM",
    "9": "PART",
    "10": "PRON",
    "11": "PROPN",
    "12": "PUNCT",
    "13": "SCONJ",
    "14": "SYM",
    "15": "VERB",
    "16": "X"
  },
  "initializer_range": 0.02,
  "label2id": {
    "ADJ": 0,
    "ADP": 1,
    "ADV": 2,
    "AUX": 3,
    "CCONJ": 4,
    "DET": 5,
    "INTJ": 6,
    "NOUN": 7,
    "NUM": 8,
    "PART": 9,
    "PRON": 10,
    "PROPN": 11,
    "PUNCT": 12,
    "SCONJ": 13,
    "SYM": 14,
    "VERB": 15,
    "X": 16
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0

## Task 4 — Training

In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load('seqeval')

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [id2label[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy']
    }


In [20]:
training_args = TrainingArguments(
    output_dir='./distilbert-ud-pos',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to='none',
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],   # or "validation" if you created it
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


## Cell 12 - Train Model

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.117439,0.128895,0.955560,0.959698,0.957625,0.964095
2,0.068093,0.112291,0.963063,0.966944,0.964999,0.970073
3,0.055450,0.113682,0.963709,0.967759,0.965730,0.970710


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2352, training_loss=0.1488429785382991, metrics={'train_runtime': 148.7391, 'train_samples_per_second': 253.007, 'train_steps_per_second': 15.813, 'total_flos': 500443158779232.0, 'train_loss': 0.1488429785382991, 'epoch': 3.0})

## Task 5 — Evaluation

In [22]:
results = trainer.evaluate(tokenized_datasets['test'])
print('Precision:', round(results['eval_precision'], 4))
print('Recall:', round(results['eval_recall'], 4))
print('F1 Score:', round(results['eval_f1'], 4))
print('Accuracy:', round(results['eval_accuracy'], 4))


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Precision: 0.9631
Recall: 0.9669
F1 Score: 0.965
Accuracy: 0.9701


In [23]:
trainer.save_model('./distilbert-ud-pos-final')
tokenizer.save_pretrained('./distilbert-ud-pos-final')
print('Model saved successfully.')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


## Task 6 — Inference

## Cell 15 — Predict on Custom Sentences

In [25]:
pos_tagger = pipeline(
    'token-classification',
    model='./distilbert-ud-pos-final',
    tokenizer='./distilbert-ud-pos-final',
    aggregation_strategy='simple'
)

sentences = [
    'John works at Google in California.',
    'The little boy is playing in the garden.',
    'My friend bought a new laptop yesterday.'
]

for sentence in sentences:
    print(' ' + '='*60)  # ← Fixed: added closing quote
    print('Input:', sentence)
    print('='*60)
    outputs = pos_tagger(sentence)
    for item in outputs:
        print(f"{item['word']:<20} -> {item['entity_group']}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Input: John works at Google in California.
john                 -> PROPN
works                -> VERB
at                   -> ADP
google               -> PROPN
in                   -> ADP
california           -> PROPN
.                    -> PUNCT
Input: The little boy is playing in the garden.
the                  -> DET
little               -> ADJ
boy                  -> NOUN
is                   -> AUX
playing              -> VERB
in                   -> ADP
the                  -> DET
garden               -> NOUN
.                    -> PUNCT
Input: My friend bought a new laptop yesterday.
my                   -> PRON
friend               -> NOUN
bought               -> VERB
a                    -> DET
new                  -> ADJ
laptop yesterday     -> NOUN
.                    -> PUNCT


## Task 7 — Comparison

| Aspect | POS Tagging | Chunking |
|---|---|---|
| Level | Grammar-level tagging | Phrase-level grouping |
| Unit | Individual word | Group of words / phrase |
| Example labels | NOUN, VERB, ADJ | B-NP, I-NP, B-VP |
| Difficulty | Easier | Medium |
| Purpose | Identify grammatical role | Identify phrase structure |

POS tagging labels each token with a grammatical category, while chunking groups tokens into phrases. POS tagging is easier and is a valid assignment option.

## Task 8 — Report / Blog

### Differences between POS Tagging and Chunking
POS tagging assigns a grammatical category to each word, such as noun, verb, adjective, or pronoun. Chunking goes beyond individual words and groups tokens into syntactic phrases like noun phrases and verb phrases.

### Challenges faced
1. The original CoNLL-2003 loading route caused dataset-script errors in Colab.
2. DistilBERT uses subword tokenization, so labels must be aligned carefully.
3. Special tokens and extra subword pieces must be assigned -100.
4. Sequence labeling needs better evaluation than plain accuracy, so seqeval is used.

### Observations and insights
- DistilBERT is lighter and faster than full BERT, which makes it suitable for Colab.
- Good preprocessing is critical for token classification tasks.
- POS tagging is simpler than chunking and easier to explain in an assignment report.
- Hugging Face Trainer makes training, evaluation, and inference much easier.

### Pipeline
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

In [ ]:
import nbformat

# Load notebook
nb = nbformat.read("Assignment_4.ipynb", as_version=4)

# Remove widget metadata
if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

# Save cleaned notebook
nbformat.write(nb, "Assignment_notebook_4.ipynb")

print("Cleaned notebook saved!")